In [406]:
import numpy as np
import pandas as pd 
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [407]:
train_data = pd.read_csv("data/train.csv")
train_data.info()
train_data.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [408]:
test_data = pd.read_csv("data/test.csv")
test_data.info()
test_data.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 418 entries, 0 to 417
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  418 non-null    int64  
 1   Pclass       418 non-null    int64  
 2   Name         418 non-null    str    
 3   Sex          418 non-null    str    
 4   Age          332 non-null    float64
 5   SibSp        418 non-null    int64  
 6   Parch        418 non-null    int64  
 7   Ticket       418 non-null    str    
 8   Fare         417 non-null    float64
 9   Cabin        91 non-null     str    
 10  Embarked     418 non-null    str    
dtypes: float64(2), int64(4), str(5)
memory usage: 36.1 KB


PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64

Let's take care of the NA values.

* Cabin has the most na in both train and test sets

    Passengers without cabin were likely in cheaper and more crowded areas which correlates with lower survival. Cabin will be transformed into a binary variable (1:Yes, 0:No). 

* Age 

    The Name column contains titles like "Mr.", "Mrs.", or "Master." These titles give us valuable clues:

    - Master usually indicates a young boy.

    - Miss usually indicates a young or unmarried woman.

    - Mr., Mrs., Dr., etc. indicate age and social status.

    Fill the missing Age values with the median age calculated from passengers who share the same Title.

* All the other NA values will just be filled with either mode of the median depending on their type.


In [409]:
train_data['HasCabin'] = train_data['Cabin'].notna().astype(int)
test_data['HasCabin'] = test_data['Cabin'].notna().astype(int)
train_data.drop('Cabin', axis=1, inplace=True)
test_data.drop('Cabin', axis=1, inplace=True)

In [410]:
def extract_title(name):
    for token in name.split():
        if token.endswith('.'): #every title ends with a '.'
            return token[:-1]  
    return 'Unknown'

train_data['Title'] = train_data['Name'].apply(extract_title)
test_data['Title'] = test_data['Name'].apply(extract_title)

# Compute medians only from training set
title_age_median = train_data.groupby('Title')['Age'].median()
global_median_age = train_data['Age'].median()

def fill_age(row):
    if pd.isna(row['Age']):
        return title_age_median.get(row['Title'], global_median_age)
    return row['Age']

train_data['Age'] = train_data.apply(fill_age, axis=1)
test_data['Age'] = test_data.apply(fill_age, axis=1)

In [411]:
#Define imputation strategy per column (median or mode)
imputation_strategy = {
    'Embarked': 'mode',   # categorical
    'Fare': 'median'      # numeric
    # other features to be added
}

#Compute the actual fill values from the training set
imputation_values = {}
for col, method in imputation_strategy.items():
    if method == 'median':
        imputation_values[col] = train_data[col].median()
    elif method == 'mode':
        imputation_values[col] = train_data[col].mode()[0]
    else:
        raise ValueError(f"Unknown method: {method}")

# Step C: Apply these values to both train and test sets
for col, value in imputation_values.items():
    train_data[col] = train_data[col].fillna(value)
    test_data[col] = test_data[col].fillna(value)

Checking if it true that all women survived, has gender_submission says.

In [412]:
women = train_data.loc[train_data.Sex == "female"]["Survived"]

rate = sum(women) / len(women) 

print(rate)


0.7420382165605095


Not all women survived. But what about men?

In [413]:
men = train_data.loc[train_data.Sex == "male"]["Survived"]

rate = sum(men)/len(men)

print(rate)

0.18890814558058924


Not all men survived as well. But the rate of survival for women is much high. gender_submission.csv was not such a bad guess. But it was only using one collumn to predict survival.

Let's try to use all the features and RandomForest.

In [414]:
y = train_data["Survived"]

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'HasCabin', 'Embarked']

X = pd.get_dummies(train_data[features])

X_test = pd.get_dummies(test_data[features])

X_test = X_test.reindex(columns=X.columns, fill_value=0)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission1.csv', index=False)

d = pd.read_csv("submission1.csv")
print(d["Survived"].sum()/len(d["Survived"]))


print(cross_val_score(model, X, y, cv=10, scoring='accuracy').mean())

0.3588516746411483
0.8282771535580524


Using all the variables, only one survided. But that is doubtfully true. Maybe I am using too many variables. Let's try and use only those variables with the biggest spearman correlation with the target survived. Spearman because it measures more than linear correlation.

In [415]:
train_data.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Embarked', 'HasCabin', 'Title'],
      dtype='str')

In [416]:
y = train_data["Survived"]

X_processed = pd.get_dummies(train_data[features])

df_corr = pd.concat([X_processed, y], axis=1)

table_corr = abs(df_corr.corr(method = "spearman"))
table_corr["Survived"].sort_values(ascending=False).head(20)

Survived      1.000000
Sex_female    0.543351
Sex_male      0.543351
Pclass        0.339668
Fare          0.323736
HasCabin      0.316912
Embarked_C    0.168240
Embarked_S    0.149683
Parch         0.138266
SibSp         0.088879
Age           0.057245
Embarked_Q    0.003650
Name: Survived, dtype: float64

The top 7 features in terms of correlation are:
* Sex
* Pclass
* Fare
* HasCabin
* Parch
* SibSp
* Age

Embarker_C and Embarked_S appear high in the table, but they are dummies of Embarker so there are a more possible values that they could have. So I choose to not see them as high because all the other Embarker dummies do not appear high in the correlation rank. 

Let's see what I get using only this features.

In [417]:
y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch", "Fare", "HasCabin", "Age", "Title"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])
X_test = X_test.reindex(columns=X.columns, fill_value=0)

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission2.csv', index=False)

d = pd.read_csv("submission2.csv")
print(d["Survived"].sum()/len(d["Survived"]))


print(cross_val_score(model, X, y, cv=10, scoring='accuracy').mean())

0.3923444976076555
0.8282771535580524


In [418]:
#Suggested by the kaggel tutorial 
y = train_data["Survived"]

features = ["Pclass", "Sex", "SibSp", "Parch"]
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission3.csv', index=False)

d = pd.read_csv("submission3.csv")
print(d["Survived"].sum()/len(d["Survived"]))


print(cross_val_score(model, X, y, cv=10, scoring='accuracy').mean())

0.35406698564593303
0.7935330836454433
